In [52]:
import sys
import json
import pandas as pd
from sentence_transformers import SentenceTransformer
from sentence_transformers import util
import numpy as np

In [53]:
with open("data/processed/chunk.json","r",encoding="utf-8") as f:
    chunks=json.load(f)

In [54]:
model = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [55]:
vec = model.encode("raise interest rates")
print(vec.shape)
print(vec[:5])

(384,)
[ 0.01950644 -0.01580445 -0.06307644 -0.04502466 -0.02718136]


In [56]:
sentences = [
    "raise interest rates",
    "increase the target range for the federal funds rate",
    "progress on vaccinations has reduced the spread of COVID-19",
]
vecs= model.encode(sentences)
print(util.cos_sim(vecs,vecs))

tensor([[1.0000, 0.4746, 0.0994],
        [0.4746, 1.0000, 0.1396],
        [0.0994, 0.1396, 1.0000]])


In [57]:
text=[chunk["text"] for chunk in chunks]
chunk_vecs =  model.encode(text,show_progress_bar=True)
print(chunk_vecs.shape)

Batches:   0%|          | 0/9 [00:00<?, ?it/s]

(260, 384)


In [58]:
np.save("data/processed/chunk_embeddings.npy", chunk_vecs)

In [65]:
def search(query,k=5):
    query_vec=model.encode(query)
    query_util=util.cos_sim(query_vec,chunk_vecs)[0]
    top = sorted(range(len(query_util)), key=lambda i: query_util[i], reverse=True)[:k]
    for i in top:
        print(round(float(query_util[i]), 2), chunks[i]["chunk_id"], chunks[i]["text"][:100])
    return [(chunks[i]["chunk_id"],float(query_util[i]))for i in top]
    

In [66]:
search("balance sheet reduction")

0.42 2021-12-15_p4_c2 The Committee judges that similar reductions in the pace of net asset purchases will likely be appro
0.4 2022-11-02_p3_c2 In addition, the Committee will continue reducing its holdings of Treasury securities and agency deb
0.4 2022-12-14_p3_c2 In addition, the Committee will continue reducing its holdings of Treasury securities and agency deb
0.39 2021-11-03_p4_c2 In light of the substantial further progress the economy has made toward the Committee's goals since
0.38 2022-05-04_p3_c1 The Committee seeks to achieve maximum employment and inflation at the rate of 2 percent over the lo


[('2021-12-15_p4_c2', 0.4218287765979767),
 ('2022-11-02_p3_c2', 0.40485137701034546),
 ('2022-12-14_p3_c2', 0.40485137701034546),
 ('2021-11-03_p4_c2', 0.38550737500190735),
 ('2022-05-04_p3_c1', 0.3757585883140564)]

In [61]:
search("inflation transitory")

0.52 2026-06-17_p4_c1 Inflation remains elevated relative to the Committee's 2 percent goal, in part reflecting supply sho
0.52 2026-07-29_p4_c1 Inflation remains elevated relative to the Committee's 2 percent goal, in part reflecting supply sho
0.52 2026-10-28_p4_c1 Inflation remains elevated. The accumulation of more information will help the Committee determine t
0.52 2026-09-16_p4_c1 Inflation remains elevated. Today's policy action will support a timelier return to the Committee's 
0.44 2024-05-01_p2_c1 The Committee seeks to achieve maximum employment and inflation at the rate of 2 percent over the lo


[('2026-06-17_p4_c1', 0.5205362439155579),
 ('2026-07-29_p4_c1', 0.5205362439155579),
 ('2026-10-28_p4_c1', 0.5179383754730225),
 ('2026-09-16_p4_c1', 0.51784747838974),
 ('2024-05-01_p2_c1', 0.4423401355743408)]

In [62]:
search("when did the Fed raise rates by 75 basis points")

0.51 2026-09-16_p2_c1 The Committee decided to raise the target range for the federal funds rate by 1/4 percentage point t
0.48 2022-05-04_p3_c1 The Committee seeks to achieve maximum employment and inflation at the rate of 2 percent over the lo
0.48 2022-01-26_p3_c1 The Committee seeks to achieve maximum employment and inflation at the rate of 2 percent over the lo
0.48 2022-09-21_p3_c1 The Committee seeks to achieve maximum employment and inflation at the rate of 2 percent over the lo
0.47 2024-09-18_p3_c1 In light of the progress on inflation and the balance of risks, the Committee decided to lower the t


[('2026-09-16_p2_c1', 0.513350784778595),
 ('2022-05-04_p3_c1', 0.4842706322669983),
 ('2022-01-26_p3_c1', 0.4800315499305725),
 ('2022-09-21_p3_c1', 0.4796648621559143),
 ('2024-09-18_p3_c1', 0.4719102084636688)]